# Permutation Feature Importance

Permutation importance measures how much a model's performance drops when a single feature's values are randomly shuffled. It is model-agnostic and computed on a held-out set, avoiding the bias of impurity-based importance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

In [ ]:
# Load Breast Cancer dataset
data = load_breast_cancer()
X = data.data
y = data.target

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train RandomForestClassifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Print test accuracy
test_accuracy = clf.score(X_test, y_test)
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
# Compute permutation importance on test set
perm_importance = permutation_importance(
    clf, X_test, y_test, n_repeats=30, random_state=42
)

# Display results
print("Permutation Importance Computed")
print(f"Mean importances shape: {perm_importance.importances_mean.shape}")

In [ ]:
# Get feature names
feature_names = data.feature_names

# Sort by mean importance (descending) and get top 10
indices = np.argsort(perm_importance.importances_mean)[::-1][:10]
top_features = [feature_names[i] for i in indices]
top_importances_mean = perm_importance.importances_mean[indices]
top_importances_std = perm_importance.importances_std[indices]

# Plot top 10 features as box plot
plt.figure(figsize=(10, 6))
y_pos = np.arange(len(top_features))
plt.barh(y_pos, top_importances_mean, xerr=top_importances_std, capsize=5)
plt.yticks(y_pos, top_features)
plt.xlabel("Permutation Importance (Mean ± Std)")
plt.title("Top 10 Features by Permutation Importance")
plt.tight_layout()
plt.show()

## Comparison with Impurity-Based Importance

Impurity-based importance (feature_importances_) is computed during training and reflects how much each feature contributes to reducing impurity, whereas permutation importance is model-agnostic and computed on held-out data.

In [ ]:
# Get impurity-based feature importances
impurity_importance = clf.feature_importances_

# Sort by impurity importance (descending) and get top 10
impurity_indices = np.argsort(impurity_importance)[::-1][:10]
impurity_features = [feature_names[i] for i in impurity_indices]
impurity_values = impurity_importance[impurity_indices]

# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot impurity-based importance
axes[0].barh(np.arange(len(impurity_features)), impurity_values, color='steelblue')
axes[0].set_yticks(np.arange(len(impurity_features)))
axes[0].set_yticklabels(impurity_features)
axes[0].set_xlabel("Impurity-Based Importance")
axes[0].set_title("Top 10 Features: Impurity-Based Importance")

# Plot permutation importance
axes[1].barh(np.arange(len(top_features)), top_importances_mean, 
              xerr=top_importances_std, capsize=5, color='coral')
axes[1].set_yticks(np.arange(len(top_features)))
axes[1].set_yticklabels(top_features)
axes[1].set_xlabel("Permutation Importance (Mean ± Std)")
axes[1].set_title("Top 10 Features: Permutation Importance")

plt.tight_layout()
plt.show()

## Interpretation

Permutation importance can differ from impurity-based importance, especially when features are correlated. Impurity-based importance is biased toward high-cardinality features, while permutation importance is computed on unseen data and reflects true predictive importance.